In [3]:
%load_ext jupyter_black

The jupyter_black extension is already loaded. To reload it, use:
  %reload_ext jupyter_black


In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F

%matplotlib inline

In [10]:
z_logits = torch.randn(2, 5)

In [11]:
results = []
for i in range(100_000):
    noise = -torch.log(-(torch.log(torch.rand_like(z_logits))))
    argmax_1hot = F.one_hot(
        (z_logits + noise).argmax(-1), num_classes=z_logits.shape[-1]
    )
    results.append(argmax_1hot)

In [13]:
z_logits = torch.randn(2, 3, requires_grad=True)
noise = -torch.log(-torch.log(torch.rand_like(z_logits)))
gumbel_softmax = torch.softmax((z_logits + noise) / 0.1, dim=-1)  # tau
gumbel_softmax

tensor([[5.7584e-16, 1.0000e+00, 2.2290e-13],
        [1.6625e-02, 3.7482e-15, 9.8337e-01]], grad_fn=<SoftmaxBackward0>)

In [ ]:
X = 
num_codes = 8
nbits_per_code = 8

In [14]:
encoder = nn.Sequential(
    nn.Linear(X.shape[-1], 256), nn.GELU(),
    nn.Linear(256, num_codes * 2**nbits_per_code)
)

decoder = nn.Sequential(
    nn.Linear(64, 256), nn.GELU(), nn.Linear(256, X.shape[1])
)

opt = torch.optim.Adam(nn.ModuleList([encoder, decoder]).parameters())

NameError: name 'X' is not defined

In [ ]:
z = F.gumbel_softmax(
    encoder(x_batch).reshape(len(x_batch), num_codes, 2**nbits_per_code),
    tau=0.1, hard=True
).flatten(1, 2) # new autoencoder
encoder(x_batch).reshape(len(x_batch), num_codes, 2**nbits_per_code).shape

In [ ]:
for i in range(1_000):
    x_batch = X[torch.randint(0, len(X), (1000,)), :]
    z = F.gumbel_softmax(encoder(x_batch), tau = 0.1, hard=True)
    loss = F.mse_loss()
    opt.zero_grad()
    loss.backward()
    opt.step()
    if (i % 10 == 0):
        print(loss.item())